In [1]:
import polars
from data_index.iceberg_config import S3TablesCatalogConfig, IcebergTableConfig
from data_index.analysis.tables import IMOS_DATA_LIVE_TABLE
from data_index.analysis.datasets import (
    DATASET,
    DATASET_FILTER,
    get_dataset_objects_df,
    get_dataset_xarray_dataset,
)

In [2]:
table = IMOS_DATA_LIVE_TABLE.load()

In [3]:
table.schema

<bound method Table.schema of live(
  1: bucket: required string (The general purpose bucket name.),
  2: key: required string (The object key name (or key) that uniquely identifies the object in the bucket.),
  3: sequence_number: required string (The sequence number, which is an ordinal that's included in the records for a given object. To order records of the same bucket and key, you can sort on sequence_number.),
  4: version_id: optional string (The object's version ID. Amazon S3 assigns a version number to objects added to the bucket.),
  5: is_delete_marker: optional boolean (The object's delete marker status. True if the object is a delete marker.),
  6: size: optional long (The object size in bytes. If is_delete_marker is True, the size is 0.),
  7: last_modified_date: optional timestamp (The object creation date or the last modified date, whichever is the latest.),
  8: e_tag: optional string (The entity tag (ETag), which is a hash of the object contents.),
  9: storage_class

In [4]:
df = table.scan(
    selected_fields=("bucket", "key", "size", "facility", "last_modified_date",),
    row_filter=(
        "facility == 'SRS'"
    ),
).to_polars()

In [5]:
dataset_df = (
    get_dataset_objects_df(
        df=df,
        dataset="station_lucinda_jetty_dalec_derived_product",
    )
)

In [6]:
ds = get_dataset_xarray_dataset(
    dataset="station_lucinda_jetty_dalec_derived_product",
)

In [11]:
nc_filenames = set(dataset_df["key"].str.split("/").list.last())
zarr_filenames = set(ds.filename.values)

In [12]:
print(len(zarr_filenames))
print(len(nc_filenames))
print(len(nc_filenames | zarr_filenames))

262
262
262


In [13]:
nc_filenames

{'IMOS_SRS-OC-LJCO_F_20160526T231553Z_LJCO_FV02_DALEC_END-20160527T025949Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160527T230902Z_LJCO_FV02_DALEC_END-20160528T025754Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160528T230942Z_LJCO_FV02_DALEC_END-20160529T025959Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160529T231022Z_LJCO_FV02_DALEC_END-20160530T025317Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160530T231059Z_LJCO_FV02_DALEC_END-20160531T025959Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160531T231137Z_LJCO_FV02_DALEC_END-20160601T031459Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160601T231214Z_LJCO_FV02_DALEC_END-20160602T031110Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160602T231250Z_LJCO_FV02_DALEC_END-20160603T032959Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160603T231325Z_LJCO_FV02_DALEC_END-20160604T032959Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160604T231359Z_LJCO_FV02_DALEC_END-20160605T034459Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160605T234128Z_LJCO_FV02_DALEC_END-20160605T234346Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160606T001346Z_LJCO_FV02_DALEC_END-20160606T010032Z.nc',
 'IMOS_SRS-OC-LJCO_F_20160706T003823Z_LJ